# Лабораторная работа № 5. Вариант 6 Романов Михаил ПМИ-3


## 1. Структура проекта

In [1]:
from pathlib import Path

ROOT = Path("lab5_6")
for p in sorted(ROOT.rglob("*")):
    if p.is_file() and ".scrapy" not in p.parts and "jobstate" not in p.parts and "httpcache" not in p.parts:
        print(p.as_posix())

lab5_6/data/crawl.db
lab5_6/data/items.jsonl
lab5_6/data/logs/c1.log
lab5_6/data/logs/c2.log
lab5_6/data/logs/d1.log
lab5_6/data/logs/d2.log
lab5_6/data/logs/r1.log
lab5_6/data/logs/r2.log
lab5_6/data/stats_c1_quotes_cache_store.json
lab5_6/data/stats_c2_quotes_cache_hit.json
lab5_6/data/stats_d1_jobdir_start.json
lab5_6/data/stats_d2_jobdir_resume.json
lab5_6/data/stats_r1_books_jobdir.json
lab5_6/data/stats_r2_books_resume.json
lab5_6/lab5_6/__init__.py
lab5_6/lab5_6/items.py
lab5_6/lab5_6/pipelines.py
lab5_6/lab5_6/settings.py
lab5_6/lab5_6/spiders/__init__.py
lab5_6/lab5_6/spiders/books.py
lab5_6/lab5_6/spiders/quotes.py
lab5_6/nocode/nocode_result.csv
lab5_6/nocode/workflow.json
lab5_6/quality/smoke_check.py
lab5_6/requirements.txt
lab5_6/scrapy.cfg


Штатная структура создана командой scrapy startproject lab5_6. Лишние файлы шаблона удалены (middlewares.py не использовался)

In [2]:
print(Path("lab5_6/lab5_6/spiders/books.py").read_text(encoding="utf-8").split("def parse_item")[1][:1200])

(self, response, title, list_price, rating):
        # карточка может быть нетипичной: все поля забираем через .get()
        # и позволяем ValidationPipeline решать, что обязательное
        item = BookItem()
        item["title"] = title
        item["list_price"] = list_price
        item["rating"] = rating
        item["list_url"] = response.url
        item["url"] = response.url
        item["source"] = "books"
        item["upc"] = response.xpath("//tr[th='UPC']/td/text()").get()
        item["price_excl_tax"] = response.xpath(
            "//tr[th='Price (excl. tax)']/td/text()"
        ).get()
        item["price_incl_tax"] = response.xpath(
            "//tr[th='Price (incl. tax)']/td/text()"
        ).get()
        item["tax"] = response.xpath("//tr[th='Tax']/td/text()").get()
        item["availability"] = response.xpath("//tr[th='Availability']/td/text()").get()
        item["num_reviews"] = response.xpath(
            "//tr[th='Number of reviews']/td/text()"
        ).get(

Карточка книги. поля берутся по заголовкам таблицы, а не по позициям, поэтому нет индексных XPath. Все поля через get(), пустое поле не роняет паучка. Уровень 1 передаёт название, цену и рейтинг через cb_kwargs.

Пайплайн из четырёх звеньев (`ITEM_PIPELINES`) для наглядности таблица:

| № | Звено | Что делает | Счётчик |
|---|---|---|---|
| 100 | `ValidatePipeline` | обязательные поля | `dropped/missing_fields` |
| 200 | `CleanPipeline` | цена → float, количество → int, дата → ISO, пробелы | `dropped/bad_value` |
| 300 | `DedupPipeline` | бизнес-ключ: UPC (книги) и sha1(автор+цитата) (цитаты) | `duplicates/<source>`, `dedup/updates` |
| 400 | `SQLitePipeline` | `executemany` пачками по 100, `INSERT OR REPLACE` / `ON CONFLICT DO UPDATE`, соединение в `open_spider`/`close_spider` | `stored/rows` |

In [3]:
import re
s = Path("lab5_6/lab5_6/settings.py").read_text(encoding="utf-8")
keys = ["ROBOTSTXT_OBEY", "USER_AGENT", "CONCURRENT_REQUESTS", "CONCURRENT_REQUESTS_PER_DOMAIN",
        "DOWNLOAD_DELAY", "DOWNLOAD_DELAY_JITTER", "AUTOTHROTTLE_ENABLED", "AUTOTHROTTLE_TARGET_CONCURRENCY",
        "DOWNLOAD_TIMEOUT", "RETRY_TIMES", "RETRY_HTTP_CODES", "DEPTH_LIMIT", "CLOSESPIDER_ITEMCOUNT",
        "HTTPCACHE_ENABLED", "LOG_LEVEL"]
for line in s.splitlines():
    if any(line.startswith(k + " ") or line.startswith(k + "=") for k in keys):
        print(line)

ROBOTSTXT_OBEY = True
USER_AGENT = "StudentScraperBot/1.0 (+https://github.com/PINNOS/parsing; educational data-harvesting course)"
CONCURRENT_REQUESTS = 8
CONCURRENT_REQUESTS_PER_DOMAIN = 4
DOWNLOAD_DELAY = 0.25
DOWNLOAD_DELAY_JITTER = 0.5
AUTOTHROTTLE_ENABLED = True
AUTOTHROTTLE_TARGET_CONCURRENCY = 2.0
DOWNLOAD_TIMEOUT = 30
RETRY_TIMES = 3
RETRY_HTTP_CODES = [429, 500, 502, 503, 504, 522, 524, 408]
DEPTH_LIMIT = 60
CLOSESPIDER_ITEMCOUNT = 350
HTTPCACHE_ENABLED = False
LOG_LEVEL = "INFO"


Значения из settings.py. Каждый параметр снабжён комментарием в файле

## 3. Прогоны и JOBDIR

Команды прогонов (выполнены в этом порядке; из корня `lab5_6/`):

```bash
scrapy crawl books -a run_id=r1_books_jobdir -s JOBDIR=jobstate/books -s CLOSESPIDER_ITEMCOUNT=100 -o data/items.jsonl
scrapy crawl books -a run_id=r2_books_resume -s JOBDIR=jobstate/books -s CLOSESPIDER_ITEMCOUNT=350 -o data/items.jsonl
scrapy crawl books -a run_id=d1_jobdir_start -a max_pages=2 -s JOBDIR=jobstate/books_demo -s CLOSESPIDER_ITEMCOUNT=25
scrapy crawl books -a run_id=d2_jobdir_resume -a max_pages=2 -s JOBDIR=jobstate/books_demo -s CLOSESPIDER_ITEMCOUNT=500
scrapy crawl quotes -a run_id=c1_quotes_cache_store -s HTTPCACHE_ENABLED=True -o data/items.jsonl
scrapy crawl quotes -a run_id=c2_quotes_cache_hit -s HTTPCACHE_ENABLED=True
```

Итоговый прогон книг: `CLOSESPIDER_ITEMCOUNT = 350`, собрано 358 записей. Первый прогон `r1` остановлен предохранителем на 100 (собрано 107, так как обработка идёт пачками).

In [4]:
import json
import glob

def load(rid):
    return json.loads(Path(f"lab5_6/data/stats_{rid}.json").read_text(encoding="utf-8"))

runs = [("r1_books_jobdir", "книги, старт"), ("r2_books_resume", "книги, resume, итог"),
        ("d1_jobdir_start", "демо JOBDIR, старт"), ("d2_jobdir_resume", "демо JOBDIR, resume"),
        ("c1_quotes_cache_store", "цитаты, кэш пишется"), ("c2_quotes_cache_hit", "цитаты, кэш читается")]
rows = []
for rid, label in runs:
    s = load(rid); st = s["stats"]
    rows.append({"прогон": rid, "описание": label, "reason": s["reason"],
                 "секунд": s["elapsed_s"], "записей": st.get("item_scraped_count", 0),
                 "запросов": st.get("downloader/request_count", 0),
                 "ответов 200": st.get("downloader/response_status_count/200", 0),
                 "ответов 404": st.get("downloader/response_status_count/404", 0),
                 "трафик, байт": st.get("downloader/response_bytes", 0),
                 "dupefilter/filtered": st.get("dupefilter/filtered", 0),
                 "httpcache/store": st.get("httpcache/store", 0),
                 "httpcache/hit": st.get("httpcache/hit", 0)})
import pandas as pd
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 20)
pd.DataFrame(rows)

,прогон,описание,reason,секунд,записей,запросов,ответов 200,ответов 404,"трафик, байт",dupefilter/filtered,httpcache/store,httpcache/hit
0,r1_books_jobdir,"книги, старт",closespider_itemcount,62.41,107,120,119,1,482392,0,0,0
1,r2_books_resume,"книги, resume, итог",closespider_itemcount,136.68,358,397,396,1,1589779,0,0,0
2,d1_jobdir_start,"демо JOBDIR, старт",closespider_itemcount,11.84,33,36,35,1,137884,0,0,0
3,d2_jobdir_resume,"демо JOBDIR, resume",finished,3.91,7,9,8,1,29039,21,0,0
4,c1_quotes_cache_store,"цитаты, кэш пишется",finished,39.39,200,211,110,1,249895,0,123,88
5,c2_quotes_cache_hit,"цитаты, кэш читается",finished,2.15,200,211,110,1,249904,0,0,211


Первый прогон остановился по лимиту записей. Второй продолжил работу из сохранённого состояния и добавил в БД 358 книг. Отдельный демонстрационный прогон показал работу фильтра повторных запросов

In [5]:
import subprocess
state = Path("lab5_6/jobstate/books_demo")
print("содержимое jobstate/books_demo:")
for p in sorted(state.rglob("*")):
    if p.is_file():
        print(" ", p.relative_to(state).as_posix(), p.stat().st_size, "байт")
print()
log = Path("lab5_6/data/logs/r2.log").read_text(encoding="utf-8")
print([l.strip() for l in log.splitlines() if "Resuming crawl" in l or "Spider closed" in l])

содержимое jobstate/books_demo:
  requests.queue/active.json 2 байт
  requests.seen 902 байт
  spider.state 5 байт

['2026-10-19 16:39:23 [scrapy.core.scheduler] INFO: Resuming crawl (134 requests scheduled)', '2026-10-19 16:41:40 [scrapy.core.engine] INFO: Spider closed (closespider_itemcount)']


 В папке состояния лежат очередь запросов и список посещённых адресов. Строка Resuming crawl подтверждает, что второй прогон продолжил первый

## 4. Проверка качества данных

Проверим число записей и уникальных ключей в crawl.db

In [6]:
import sqlite3
con = sqlite3.connect("lab5_6/data/crawl.db")
q = lambda sql: pd.read_sql(sql, con)
summary = pd.DataFrame({
    "таблица": ["books", "quotes"],
    "записей": [q("SELECT COUNT(*) n FROM books").n[0], q("SELECT COUNT(*) n FROM quotes").n[0]],
    "уникальных ключей": [q("SELECT COUNT(DISTINCT upc) n FROM books").n[0],
                          q("SELECT COUNT(DISTINCT business_key) n FROM quotes").n[0]],
})
summary

,таблица,записей,уникальных ключей
0,books,491,491
1,quotes,100,100


Дублей по ключам нет. В БД 491 книга, а в `items.jsonl`  465. Остальные записи попали в БД во время отладочных и демонстрационных прогонов

что там полноте полей

In [7]:
def completeness(table, cols):
    total = q(f"SELECT COUNT(*) n FROM {table}").n[0]
    out = []
    for c in cols:
        filled = q(f"SELECT COUNT(*) n FROM {table} WHERE {c} IS NOT NULL AND {c} <> ''").n[0]
        out.append({"таблица": table, "поле": c, "заполнено, %": round(100 * filled / total, 1)})
    return pd.DataFrame(out)

pd.concat([
    completeness("books", ["upc", "title", "price_excl_tax", "availability", "quantity",
                           "num_reviews", "category", "rating", "description"]),
    completeness("quotes", ["quote", "author", "born_date", "born_location", "biography", "tags"]),
], ignore_index=True)

,таблица,поле,"заполнено, %"
0,books,upc,100.0
1,books,title,100.0
2,books,price_excl_tax,100.0
3,books,availability,100.0
4,books,quantity,100.0
5,books,num_reviews,100.0
6,books,category,100.0
7,books,rating,100.0
8,books,description,99.8
9,quotes,quote,100.0


 Обязательные поля заполнены полностью. У одной книги нет необязательного описания. Данные об авторе есть у всех 100 цитат.

Посмотрим причины отброса и повторы

In [8]:
keys = ["dropped/missing_fields", "dropped/bad_value", "duplicates/books", "duplicates/quotes", "dedup/updates", "stored/rows"]
agg = {k: 0 for k in keys}
for rid in ["r1_books_jobdir", "r2_books_resume", "c1_quotes_cache_store", "c2_quotes_cache_hit", "d1_jobdir_start", "d2_jobdir_resume"]:
    st = load(rid)["stats"]
    for k in keys:
        agg[k] += st.get(k, 0)
pd.DataFrame([{"счётчик": k, "сумма по прогонам": v} for k, v in agg.items()])

,счётчик,сумма по прогонам
0,dropped/missing_fields,0
1,dropped/bad_value,0
2,duplicates/books,0
3,duplicates/quotes,0
4,dedup/updates,200
5,stored/rows,905


Отброшенных записей нет. Счётчик dedup/updates равен 200, данные цитат дополнялись при загрузке страниц авторов. Повторы книг между прогонами обновляются в БД.

распределение рейтингов и другие показатели:

In [9]:
ratings = q("SELECT rating, COUNT(*) AS книг FROM books GROUP BY rating ORDER BY rating")
by_cat = q("""SELECT category, COUNT(*) AS книг, ROUND(AVG(price_excl_tax), 2) AS средняя_цена_без_налога
              FROM books GROUP BY category ORDER BY книг DESC LIMIT 8""")
authors = q("SELECT author, COUNT(*) AS цитат, COUNT(born_date) AS с_датой FROM quotes GROUP BY author ORDER BY цитат DESC LIMIT 5")
print("рейтинг (1..5):"); print(ratings.to_string(index=False)); print()
print("средняя цена по категориям:"); print(by_cat.to_string(index=False)); print()
print("авторы цитат:"); print(authors.to_string(index=False))

рейтинг (1..5):
 rating  книг
      1   119
      2    85
      3    99
      4    82
      5   106

средняя цена по категориям:
      category  книг  средняя_цена_без_налога
       Default    71                    34.64
    Nonfiction    46                    31.67
Sequential Art    45                    36.57
       Fiction    32                    35.38
   Young Adult    27                    35.65
       Fantasy    25                    37.42
     Childrens    22                    33.94
 Add a comment    21                    32.74

авторы цитат:
         author  цитат  с_датой
Albert Einstein     10       10
   J.K. Rowling      9        9
 Marilyn Monroe      7        7
     Mark Twain      6        6
      Dr. Seuss      6        6


Рейтинги распределены примерно равномерно. Средняя цена по показанным категориям от 31 до 37 фунтов

трафик и коды ответов:

In [10]:
resp = []
for rid, _ in runs:
    st = load(rid)["stats"]
    resp.append({"прогон": rid, "200": st.get("downloader/response_status_count/200", 0),
                 "404": st.get("downloader/response_status_count/404", 0),
                 "МБ": round(st.get("downloader/response_bytes", 0) / 1e6, 3),
                 "повторов (retry)": st.get("retry/count", 0),
                 "ошибок сети": st.get("network/errors", 0)})
pd.DataFrame(resp)

,прогон,200,404,МБ,повторов (retry),ошибок сети
0,r1_books_jobdir,119,1,0.482,0,0
1,r2_books_resume,396,1,1.590,0,0
2,d1_jobdir_start,35,1,0.138,0,0
3,d2_jobdir_resume,8,1,0.029,0,0
4,c1_quotes_cache_store,110,1,0.250,0,0
5,c2_quotes_cache_hit,110,1,0.250,0,0


В каждом прогоне есть один ответ 404.... для robots.txt. Повторов запросов и сетевых ошибок не было. Прогон `r2` передал 1,59 МБ за 397 запросов.

## 5. Smoke-check деградации

quality/smoke_check.py сравнивает два прогона. Предупреждение появляется, если число записей упало больше чем на 20 % или доля пустых обязательных полей в БД выше 5 %. Первый аргумент текущий прогон, второй предыдущий.

In [11]:
import os, sys
env = dict(os.environ, PYTHONIOENCODING="utf-8")
for cur, prev in [("data/stats_r2_books_resume.json", "data/stats_r1_books_jobdir.json"),
                  ("data/stats_c2_quotes_cache_hit.json", "data/stats_c1_quotes_cache_store.json")]:
    res = subprocess.run([sys.executable, "quality/smoke_check.py", cur, prev],
                         cwd="lab5_6", capture_output=True, text=True, encoding="utf-8", env=env)
    print(f"== {cur} vs {prev}, exit={res.returncode}")
    print(res.stdout.strip()); print()

== data/stats_r2_books_resume.json vs data/stats_r1_books_jobdir.json, exit=0
записей: текущий 358, предыдущий 107, падение -234.6%
пусто books.upc: 0.0%
пусто books.title: 0.0%
пусто books.price_excl_tax: 0.0%
пусто books.availability: 0.0%
пусто books.category: 0.0%
пусто quotes.quote: 0.0%
пусто quotes.author: 0.0%
пусто quotes.author_url: 0.0%
OK: деградации не обнаружено

== data/stats_c2_quotes_cache_hit.json vs data/stats_c1_quotes_cache_store.json, exit=0
записей: текущий 200, предыдущий 200, падение 0.0%
пусто books.upc: 0.0%
пусто books.title: 0.0%
пусто books.price_excl_tax: 0.0%
пусто books.availability: 0.0%
пусто books.category: 0.0%
пусто quotes.quote: 0.0%
пусто quotes.author: 0.0%
пусто quotes.author_url: 0.0%
OK: деградации не обнаружено



Оба сравнения прошли без предупреждений. При обратном порядке аргументов число записей падает на 70 %, и проверка возвращает код 1. Неверные, но непустые значения такая проверка не заметит.

## 6. n8n



![n8n run](nocode/screen_run.png)

In [12]:
csv = pd.read_csv("lab5_6/nocode/nocode_result.csv", encoding="utf-8")
print("строк в CSV:", len(csv), "| уникальных UPC:", csv["upc"].nunique(), "| пустых полей:", int(csv.isna().sum().sum()))
print("диапазон quantity:", csv["quantity"].describe()[["min", "max"]].to_dict())
csv.head(3)

строк в CSV: 30 | уникальных UPC: 30 | пустых полей: 0
диапазон quantity: {'min': 18.0, 'max': 22.0}


,upc,title,category,price_excl_tax,price_incl_tax,tax,price_list,availability,quantity,num_reviews,url,source
0,a897fe39b1053632,A Light in the Attic,Poetry,51.77,51.77,0,51.77,In stock (22 available),22,0,https://books.toscrape.com/catalogue/a-light-i...,n8n
1,90fa61229261140a,Tipping the Velvet,Historical Fiction,53.74,53.74,0,53.74,In stock (20 available),20,0,https://books.toscrape.com/catalogue/tipping-t...,n8n
2,6957f44c3847a760,Soumission,Fiction,50.10,50.10,0,50.10,In stock (20 available),20,0,https://books.toscrape.com/catalogue/soumissio...,n8n


Из 40 ссылок выбраны 30. Все карточки собраны, после удаления дублей осталось 30 строк. Количество книг взято из текста наличия


## 7. Сравнение подходов (измеренные значения)

Время разработки не замерял. Остальные значения получены из статистики и execution

In [13]:
r2 = load("r2_books_resume")
n8n_secs = 46.3
cmp = pd.DataFrame([
    {"метрика": "Время разработки, мин", "Scrapy": "не измерялось", "n8n": "не измерялось"},
    {"метрика": "Записей собрано", "Scrapy": f"{r2['stats']['item_scraped_count']} (книги, прогон r2)", "n8n": len(csv)},
    {"метрика": "Время прогона, с", "Scrapy": r2["elapsed_s"], "n8n": n8n_secs},
    {"метрика": "Записей в минуту", "Scrapy": round(358 / r2["elapsed_s"] * 60, 1), "n8n": round(len(csv) / n8n_secs * 60, 1)},
    {"метрика": "Число сетевых запросов", "Scrapy": r2["stats"]["downloader/request_count"], "n8n": "32 (2 списка + 30 карточек)"},
    {"метрика": "Трафик, МБ", "Scrapy": round(r2["stats"]["downloader/response_bytes"] / 1e6, 2), "n8n": "не измерялся"},
    {"метрика": "Дедупликация запросов", "Scrapy": "dupefilter (21 в демо)", "n8n": "нет"},
    {"метрика": "Возобновление после сбоя", "Scrapy": "JOBDIR (проверено)", "n8n": "нет, перезапуск с начала"},
    {"метрика": "Расписание и оповещения", "Scrapy": "внешний cron, без оповещений", "n8n": "Schedule Trigger, ветка ошибки (без мессенджера)"},
    {"метрика": "Версионирование логики", "Scrapy": "git (код)", "n8n": "история версий workflow в n8n"},
    {"метрика": "Оценка усилий на поддержку (1–5)", "Scrapy": 4, "n8n": 2},
])
cmp

,метрика,Scrapy,n8n
0,"Время разработки, мин",не измерялось,не измерялось
1,Записей собрано,"358 (книги, прогон r2)",30
2,"Время прогона, с",136.68,46.3
3,Записей в минуту,157.2,38.9
4,Число сетевых запросов,397,32 (2 списка + 30 карточек)
5,"Трафик, МБ",1.59,не измерялся
6,Дедупликация запросов,dupefilter (21 в демо),нет
7,Возобновление после сбоя,JOBDIR (проверено),"нет, перезапуск с начала"
8,Расписание и оповещения,"внешний cron, без оповещений","Schedule Trigger, ветка ошибки (без мессенджера)"
9,Версионирование логики,git (код),история версий workflow в n8n


Scrapy собрал около 157 записей в минуту, n8n  около 39. В n8n проще видеть логику на схеме. Scrapy удобнее для больших обходов благодаря сохранению состояния и настройкам запросов. Оценка удобства основана на личном опыте :)
